# Feature Engineering on The Car Price Dataset

## Introduction:
This is Phase 2 of the project, continuing from where the EDA (Phase 1) left off. The goal is to make a clean and ready-made dataset for the Machine Learning model
This notebook will pay special heed to converting the columns into informative features so that it runs smoothly in future ML models

## Objectives:
- Recreate the cleaned dataset from Phase 1
- Remove redundant/uninformative features identified in EDA
- Encode categorical variables for ML compatibility
- Extract features from the options column
- Output a final ML-ready dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("../data/car_price_dataset.csv")

***
# Cleaning 

__Note: Cleaning is exactly the same as done in first phase of the project__

### Making all the columns lower case and removing extra spaces

In [3]:
df.columns=(df.columns.str.lower().str.strip())

### Renaming of the columns

In [4]:
df = df.rename(columns={"enginesize(l)": "enginesize"})
df = df.rename(columns={"fuelefficiency(l/100km)": "fuelefficiency"})
df = df.rename(columns={"price($)": "price"})
df = df.rename(columns={"mileage(km)": "mileage"})

### Defining Scope
- We are removing rows with missing priceperkm because we only want to deal with old cars.
- Since out of 2,459 missing priceperkm values, 2,454 belong to New cars, and only 5 belong to 
- Used cars, that is why we are dropping these rows, which will remove all New cars (out of scope)
- We also removed the rows having fuel type as electric, since we only wanted to analyze cars running on fuels.
- This led to shrinking our 50000 rows into 42708 rows.

In [5]:
df=df.dropna(subset="priceperkm")

In [6]:
df = df[df["fueltype"] != "Electric"]

### Using mapping dictionary to change simple binary values into booleans

In [7]:
bool_map={'Yes':True,'No':False}
bool_map2={'Valid':True,'Expired':False}
bool_map3={'Incomplete':True,'Complete':False}
df["accidenthistory"]=df["accidenthistory"].map(bool_map)
df["insurance"]=df["insurance"].map(bool_map2)
df["registrationstatus"]=df["registrationstatus"].map(bool_map3)

## Final Cleansed Data

In [8]:
pd.set_option('display.max_columns', None)
df.head(20)

,brand,model,year,carage,condition,mileage,enginesize,fueltype,horsepower,torque,transmission,drivetype,bodytype,doors,seats,color,interior,options,city,accidenthistory,insurance,registrationstatus,fuelefficiency,priceperkm,price
0,Porsche,Panamera,2008,17,Used,256395,3.3,Gasoline,513,395,Manual,AWD,Sedan,4,4,White,Cloth,"Navigation, Cruise Control, Heated Seats, Blue...",Tehran,False,True,True,11.96,0.05,13884
1,Audi,A6,2023,2,Used,20433,2.2,Diesel,302,270,Manual,FWD,Sedan,4,5,Black,Cloth,"Parking Sensors, Cruise Control, Touchscreen",Berlin,True,False,True,8.74,1.90,38888
2,BMW,X5,2022,3,Used,52328,3.2,Gasoline,400,388,Automatic,AWD,SUV,5,7,Gray,Leather,"Touchscreen, Bluetooth, Cruise Control, Naviga...",Tokyo,True,True,False,15.68,0.63,33074
3,Hyundai,Tucson,2019,6,Used,91878,1.6,Hybrid,187,219,Automatic,FWD,SUV,5,5,Silver,Cloth,"Sunroof, Rear Camera, Bluetooth, Parking Senso...",Delhi,False,False,False,9.45,0.14,12966
4,Fiat,500,2012,13,Damaged,192331,1.1,Gasoline,90,112,Automatic,FWD,Hatchback,3,4,Red,Leather,"Heated Seats, Touchscreen",Delhi,False,True,False,7.16,0.01,2670
5,Porsche,911 Carrera,2018,7,Used,110968,3.3,Gasoline,565,392,Automatic,AWD,Coupe,2,2,White,Cloth,Sunroof,Paris,False,False,True,12.69,0.43,47830
6,Mercedes-Benz,S-Class,2019,6,Used,82607,5.8,Hybrid,509,750,Manual,RWD,Sedan,4,5,Silver,Cloth,"Navigation, Parking Sensors, Touchscreen",Los Angeles,True,False,False,17.83,0.62,51189
7,Porsche,Panamera,2014,11,Used,163074,3.5,Gasoline,513,428,Manual,RWD,Sedan,4,4,Red,Cloth,"Bluetooth, Touchscreen, Cruise Control, Heated...",Tokyo,False,True,False,13.06,0.17,27296
8,Audi,Q7,2007,18,Used,274471,2.0,Gasoline,315,277,Automatic,AWD,SUV,5,7,Red,Leather,"Touchscreen, Bluetooth",Tokyo,True,False,True,10.38,0.02,5389
9,Ford,Mustang,2019,6,Used,95498,4.6,Gasoline,364,588,Manual,RWD,Coupe,2,2,Red,Cloth,"Touchscreen, Cruise Control, Navigation, Bluet...",Berlin,True,True,False,14.90,0.11,10704


***
# Dropping Uninformative Features

## Transmission, Insurance, Interior, Color, City, Registration:
- From our first phase of data analysis, we came to conclude that all of these features carry little to no change in dataset information 
- Implying that we are better off dropping them, as they will be uninformative for our ML model

In [9]:
df = df.drop(columns=["transmission"])
df = df.drop(columns=["insurance"])
df = df.drop(columns=["interior"])
df = df.drop(columns=["color"])
df = df.drop(columns=["city"])
df = df.drop(columns=["registrationstatus"])

***
# Data Leakage causing Feature

### Price per Km
The "Price per Km" feature is a strong candidate for removal from the dataset because it poses a risk of data leakage. This feature is directly derived from our target variable, price (price/mileage), meaning it contains information about the answer we're trying to predict. A model trained with this feature could reconstruct price almost exactly using a simple mathematical shortcut, but in a real-world scenario, price per km wouldn't be available before price itself is known. This would make the model appear highly accurate during training while being unusable for genuine prediction. Therefore, removing this feature is advisable to ensure honest, generalizable results.

In [10]:
df = df.drop(columns=["priceperkm"])

***
# Multicollinearity

__Note:__ In Multicolinearity we will drop the columns that are different although carries the same information meaning. Implying that they the are highly correlated to each other

### Mileage and Car Age Redundancy:
- Mileage and car age are both highly correlated with each other, as discovered in Data Analysis. Due to this strong correlation, these two features carry the same information, meaning which implies that even if one of them is dropped, there would be nearly no effect on the dataset
- Between the two, I picked Car age as a column to be dropped and Mileage to be retained because the sequential change of Mileage over the years and its vast range of numbers will allow more learning
chances for our ML model than the Car age would ever be able to

In [11]:
df = df.drop(columns=["carage"])

### Engine Size and Torque Redundancy:

- Engine size and torque are highly correlated (0.97), meaning they carry largely overlapping information about engine performance.
- Between the two, I picked engine size to retain and torque to drop, because engine size (e.g., "2.0L") acts as a common real-world indicator of a car's class/segment,
which is more directly meaningful than a raw torque figure.

In [12]:
df = df.drop(columns=["torque"])

***
# Encoding

## Labelled Encoding

__Note:__ Accident History has already been encoded via boolean where cars with accident are enocded as 1 and those with no accident is 0

### Condition
- Condition contains two unique features: Damaged and Used 
- Damaged can be put in the category of worse than Used; that is why we are assigning 1 to Used and 0 to Damaged

In [13]:
df["condition"].unique()

<StringArray>
['Used', 'Damaged']
Length: 2, dtype: str

In [14]:
condition_map = {"Used": 0, "Damaged": 1}
df["condition"] = df["condition"].map(condition_map)

__Note:__ 1 represents the "flagged" state (Damaged, Accident History present) 
rather than being tied to whether price is higher or lower — this keeps 
the convention consistent with how accidenthistory was encoded in Phase 1.

## One Hot Encoding

The categorical columns listed below do not have a natural order and will be one-hot encoded: `brand`, `fueltype`, `drivetype`, and `bodytype`. In one-hot encoding, each category of a feature will be converted into a boolean format. One category of each feature is dropped to avoid the "dummy variable trap" — if all categories were kept, one column would always be predictable from the others (e.g., if not Gasoline and not Hybrid, it must be Diesel), creating redundant, perfectly correlated information. This is the same multicollinearity concern discussed earlier, just applied to categorical encodings.

### Note on the Model

- The model has 40 unique values, which would result in 39 new columns if one-hot encoded. In contrast, there are only 15 columns for the brand.
- The brand category encapsulates broader market positioning. Encoding every individual model might lead to overfitting, as specific models may have fewer rows, while significantly increasing dimensionality.
- Therefore, the decision is to drop the model category and retain the brand as the broader indicator.

In [15]:
df = df.drop(columns=["model"])

In [16]:
df = pd.get_dummies(df, columns=["brand", "fueltype", "drivetype", "bodytype"], drop_first=True)

***
# Feature Extraction

## Feature Extraction from the Option Column
The option column contains comma-separated values, such as "Bluetooth," "Sunroof," and others. To determine if any of these values affect the prices, we will extract them. If they do impact the prices, we will create separate columns for each value, indicating with "True" or "False" whether the item is present.

In [17]:
options_list = ["Navigation", "Cruise Control", "Heated Seats", "Parking Sensors", 
                "Bluetooth", "Touchscreen", "Sunroof", "Rear Camera"]
for option in options_list:
    df[f"has_{option.lower().replace(' ', '_')}"] = df["options"].str.contains(option)

In [18]:
for option in options_list:
    col = f"has_{option.lower().replace(' ', '_')}"
    print(df.groupby(col)["price"].median())

has_navigation
False    9079.0
True     9218.0
Name: price, dtype: float64
has_cruise_control
False    9009.0
True     9337.0
Name: price, dtype: float64
has_heated_seats
False    9057.0
True     9246.0
Name: price, dtype: float64
has_parking_sensors
False    9156.0
True     9101.0
Name: price, dtype: float64
has_bluetooth
False    9120.0
True     9158.0
Name: price, dtype: float64
has_touchscreen
False    9115.0
True     9175.0
Name: price, dtype: float64
has_sunroof
False    9105.0
True     9175.0
Name: price, dtype: float64
has_rear_camera
False    9089.5
True     9209.5
Name: price, dtype: float64


- All 8 options appear in a nearly identical share of cars (~15,900-16,100 out of 42,708)
- This confirms that it has little to no impact on the price whether a car has or does not have these features. 
- Most of these price variations are as negligible as factors like interior color and other uninformative features, which have led to their removal.
- Therefore, it is safe to eliminate this column, as it does not provide any valuable information that affects prices.

In [19]:
df = df.drop(columns=["options"] + [f"has_{opt.lower().replace(' ', '_')}" for opt in options_list])

***
# Final Information and Dataset

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 42708 entries, 0 to 49999
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year                 42708 non-null  int64  
 1   condition            42708 non-null  int64  
 2   mileage              42708 non-null  int64  
 3   enginesize           42708 non-null  float64
 4   horsepower           42708 non-null  int64  
 5   doors                42708 non-null  int64  
 6   seats                42708 non-null  int64  
 7   accidenthistory      42708 non-null  bool   
 8   fuelefficiency       42708 non-null  float64
 9   price                42708 non-null  int64  
 10  brand_BMW            42708 non-null  bool   
 11  brand_Chevrolet      42708 non-null  bool   
 12  brand_Dacia          42708 non-null  bool   
 13  brand_Fiat           42708 non-null  bool   
 14  brand_Ford           42708 non-null  bool   
 15  brand_Honda          42708 non-null  bool   
 16  br

In [21]:
df.shape

(42708, 34)

In [22]:
pd.set_option('display.max_columns', None)
df.head(10)

,year,condition,mileage,enginesize,horsepower,doors,seats,accidenthistory,fuelefficiency,price,brand_BMW,brand_Chevrolet,brand_Dacia,brand_Fiat,brand_Ford,brand_Honda,brand_Hyundai,brand_Kia,brand_Mazda,brand_Mercedes-Benz,brand_Peugeot,brand_Porsche,brand_Renault,brand_Toyota,brand_Volkswagen,fueltype_Gasoline,fueltype_Hybrid,drivetype_FWD,drivetype_RWD,bodytype_Coupe,bodytype_Hatchback,bodytype_Pickup,bodytype_SUV,bodytype_Sedan
0,2008,0,256395,3.3,513,4,4,False,11.96,13884,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True
1,2023,0,20433,2.2,302,4,5,True,8.74,38888,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True
2,2022,0,52328,3.2,400,5,7,True,15.68,33074,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False
3,2019,0,91878,1.6,187,5,5,False,9.45,12966,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,True,False
4,2012,1,192331,1.1,90,3,4,False,7.16,2670,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,False,False
5,2018,0,110968,3.3,565,2,2,False,12.69,47830,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False
6,2019,0,82607,5.8,509,4,5,True,17.83,51189,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,True
7,2014,0,163074,3.5,513,4,4,False,13.06,27296,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,True
8,2007,0,274471,2.0,315,5,7,True,10.38,5389,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False
9,2019,0,95498,4.6,364,2,2,True,14.90,10704,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,True,False,False,False,False


***
# Exporting New CSV File

In [23]:
df.to_csv("../data/car_price_features.csv", index=False)

***
# Summary and Key Decisions

This notebook utilizes the cleaned dataset from the first phase (Exploratory Data Analysis). Based on the analysis conducted in the previous notebook, we performed several steps for feature engineering, resulting in a cleaned dataset that now contains 42,708 rows and 34 columns.

### Dropping Unimportant Features
From the initial phase of the notebook, we concluded that features such as Transmission, Insurance, Interior, Color, City, and Registration have little to no effect on prices; therefore, we dropped them during feature engineering.

### Data Leakage Causing Feature
Upon analysis, I suspected that "Price per Km" is a potential data leakage feature, as it is directly linked to our target variable and can be derived from price/mileage. Consequently, we decided to drop it.

### Multicollinearity
- __Mileage and Car Age__: These two features are strongly correlated and provide similar information. We chose to drop "Car Age" mainly because the distribution of "Mileage" has a much wider range of values compared to "Car Age."
- __Engine Size and Torque__: These two features are also strongly correlated. We opted to drop "Torque," as "Engine Size" is a more standard metric for evaluating a car in the real world and is more familiar to the average consumer.

### Encoding
- __Label Encoding__ was performed on the "Condition" feature because its values carry significant weight, with "Used" cars being more valued than "Damaged" cars. (Used cars assigned 0, and Damaged cars assigned 1)
- __One-Hot Encoding__ was applied to the features: Brand, Fuel Type, Body Type, and Drive Type. These features do not have an inherent order, so we created separate columns for each value. It’s worth noting that the "Model" column, which contained 40 unique values specific to particular `Note:` brands, was dropped, as it provided no meaningful information and had 40 different values. I preferred to drop this column rather than create 40 additional ones.

### Feature Extraction
We conducted feature extraction on the "Options" column, where we identified 8 different words. However, since these words had little to no effect on pricing, we dropped all of them along with the original "Options" column.

### Kept As-Is
The following features were retained in their original form: `year`, `mileage`, `enginesize`, `horsepower`, `doors`, `seats`, `accidenthistory`, `fuelefficiency`, and `price`, as they are either already numeric/boolean or confirmed to have a meaningful relationship with price.

### Final Dataset
The final dataset consists of 42,708 rows and 34 columns, all formatted as numeric or boolean, and has been exported as `car_price_features.csv`.

***
# Future Work

- __Scaling numeric features__: (deferred until you know which model you're using, since tree-based models don't need it but linear/KNN do)
- __Building a model__: using this dataset to predict prices of the cars
- __Feature importance review__: once a model exists, check whether any currently-kept features (e.g., `doors`, `seats`, `drivetype`) actually contribute meaningfully, and revisit borderline categorical checks (`bodytype` vs `drivetype` association) if needed.
- __Options revisit__: if a future dataset version includes more complete or higher-quality options data, re-test whether specific features (e.g., sunroof, navigation) show a meaningful price relationship.